# SimPro-ValueNetwork — Prepare a SMILES Dataset

Generate 17 property labels from complete SMILES for fine-tuning preparation. This example uses the SMILES model and has no fine-tuning loop.

Enable enumeration of unspecified stereochemistry when completeness is uncertain. This expands candidates rather than determining experimental stereo.
Every retained molecule undergoes 3D optimization using CPU RDKit MMFF94; no NVIDIA runtime is required.

Tokenizer, mapping, and scaler remain fixed. Start with the built-in samples, then select a custom CSV.
See docs/PROPERTIES.md for definitions and units.

Optional learning example; pretrained inference does not require this notebook.

Extract the project to `/content/SimPro-ValueNetwork`, or set `PROJECT_DIR` to your project root. Google Drive is optional: enable `USE_GOOGLE_DRIVE` and select your mounted project path. Checkpoints may be stored at any readable location.


The project is SimPro-ValueNetwork; the distribution/import name is simpro_vn, with simpro-vn and simpro-vn-train console entrypoints. Use normal installation in a fresh runtime. Use [Comprehensive Validation](colab_comprehensive_validation.ipynb) for all functional checks in one run.


In [ ]:
USE_GOOGLE_DRIVE = False  # Enable only if your files are stored in Google Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import sys
import json
import hashlib

PROJECT_DIR = Path("/content/SimPro-ValueNetwork")
INPUT_CSV = None  # Set (PROJECT_DIR / "data/my_smiles.csv"); None uses built-in samples.
SMILES_COLUMN = "SMILES"
MODEL_SETTING = 5000  # 5000, 10000, 20000, 50000, or "no_bpe"
DEPTH = 6
stem = "smiles_no_bpe" if MODEL_SETTING == "no_bpe" else f"smiles_bpe_min_freq_{MODEL_SETTING}"
CHECKPOINT_PATH = PROJECT_DIR / "weights" / "smiles" / f"value_network_best_{stem}_d{DEPTH}.pt"
OUTPUT_DIR = PROJECT_DIR / "prepared_datasets" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

ENUMERATE_STEREO = True
MAX_ISOMERS = 32
EMBEDDING_SEED = 42
MAX_ITERATIONS = 1000
VAL_FRACTION = 0.2
MAX_INPUT_ROWS = 100  # Read at most 100 rows by default; set None explicitly to process the full CSV.

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
assert (PROJECT_DIR / "examples/finetuning_data.py").is_file(), "Sync the latest project"
print("Output:", OUTPUT_DIR)


In [ ]:
%pip install "{PROJECT_DIR}[data]"


---

## 1 Load the existing model and fixed assets

Choose the pretrained model to be fine-tuned later. Asset paths may change, but tokenizer, mapping, and scaler must match the selected bundled contents. Do not refit the scaler or redefine embedding token IDs with a new tokenizer. This cell loads model/assets; later checks validate encoding, lengths, round trips, and standardization per row.


In [ ]:
from simpro_vn import ValueNetworkPredictor

predictor = ValueNetworkPredictor(
    checkpoint_path=CHECKPOINT_PATH,
    representation="smiles",
    min_frequency=MODEL_SETTING,
    depth=DEPTH,
    device="cpu",  # Data compatibility checks do not require a GPU.
)
print("Bundle fingerprint:", predictor.bundle_fingerprint)
print("Maximum encoded length including BOS/EOS:", predictor.max_seq_len)


In [ ]:
import importlib.util
import importlib.metadata
import pandas as pd

spec = importlib.util.spec_from_file_location("finetuning_data_helpers", PROJECT_DIR / "examples/finetuning_data.py")
data_helpers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_helpers)

if INPUT_CSV is None:
    input_data = pd.DataFrame({SMILES_COLUMN: [
        "CCO", "CC(O)C(=O)O", "N[C@@H](C)C(=O)O", "c1ccccc1",
    ]})
else:
    INPUT_CSV = Path(INPUT_CSV)
    input_data = pd.read_csv(INPUT_CSV, nrows=MAX_INPUT_ROWS)
print("Rows read:", len(input_data))
display(input_data.head())

versions = {name: importlib.metadata.version(name) for name in ["torch", "rdkit", "tokenizers", "pandas", "freesasa"]}
metadata = {
    "input_csv": str(INPUT_CSV), "input_rows_read": len(input_data), "max_input_rows": MAX_INPUT_ROWS,
    "enumerate_stereo": ENUMERATE_STEREO, "max_isomers": MAX_ISOMERS,
    "embedding_seed": EMBEDDING_SEED, "max_iterations": MAX_ITERATIONS,
    "optimization_backend": "rdkit_cpu_mmff94", "model_setting": MODEL_SETTING, "depth": DEPTH,
    "bundle_fingerprint": predictor.bundle_fingerprint, "checkpoint_path": str(CHECKPOINT_PATH),
    "versions": versions,
    "freesasa_defaults": data_helpers.freesasa.Parameters.defaultParameters,
}
(OUTPUT_DIR / "preparation_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


---

## 2 Validate SMILES, enumerate unspecified stereo, and deduplicate

Use onlyUnassigned=True instead of intentionally flipping known absolute stereo. tryEmbedding=True is an enumeration heuristic, not the later 3D optimization. max_isomers is a per-input cap, not proof of exhaustive enumeration; reaching it sets Possibly_Capped. A shared connectivity parent uses one Parent_Group for later Train/Val splitting.


In [ ]:
expanded, enumeration_rejections = data_helpers.expand_smiles(
    input_data, SMILES_COLUMN, enumerate_stereo=ENUMERATE_STEREO, max_isomers=MAX_ISOMERS,
)
expanded.to_csv(OUTPUT_DIR / "expanded_smiles.csv", index=False)
enumeration_rejections.to_csv(OUTPUT_DIR / "enumeration_rejections.csv", index=False)
print("After expansion and deduplication:", len(expanded), "; rejected/duplicate:", len(enumeration_rejections))
assert not expanded.empty, "No molecules remain; inspect rejection records"
display(expanded.head())


---

## 3 Optimize every retained molecule in 3D

Use AddHs, EmbedMolecule(seed=42), and MMFF94(maxIters=1000). Accept only convergence status 0, checking energy, coordinates, and canonical isomeric SMILES. Log parsing, embedding, parameter, convergence, or structure-change failures. Molecules without stereocenters are also optimized. Predictions never substitute for calculated labels. This is a single-conformer workflow, not a claim of global minima or complete geometry screening.


In [ ]:
optimized, optimization_rejections = data_helpers.optimize_all(
    expanded, seed=EMBEDDING_SEED, max_iterations=MAX_ITERATIONS,
)
optimization_rejections.to_csv(OUTPUT_DIR / "optimization_rejections.csv", index=False)
print("Converged:", len(optimized), "; failed:", len(optimization_rejections))
assert not optimized.empty, "No successful conformers; inspect optimization_rejections.csv"
optimized.to_pickle(OUTPUT_DIR / "optimized_conformers.pkl")


---

## 4 Calculate 17 properties from those optimized conformers

2D descriptors use the RemoveHs view; 3D descriptors use the explicit-hydrogen conformer. SNDM is the Gasteiger dipole magnitude times 4.803204 divided by heavy-atom count. Invalid individual charges become zero and their number is logged. SASA uses RDKit vdW radii and installed FreeSASA defaults, with N/O polar and hydrogens nonpolar. Labels stay on their original scale; no scaler is fitted. Any missing/nonfinite label excludes the row from ready data.


In [ ]:
labeled, property_rejections = data_helpers.calculate_all(optimized)
property_rejections.to_csv(OUTPUT_DIR / "property_rejections.csv", index=False)
labeled.to_csv(OUTPUT_DIR / "labeled_raw.csv", index=False)
print("All 17 properties completed:", len(labeled), "; failed:", len(property_rejections))
assert not labeled.empty, "No complete labels; inspect property_rejections.csv"
display(labeled[["NEW_Isomeric_SMILES", *data_helpers.PROPERTY_NAMES]].head())


---

## 5 Check model tokenizer, mapping, and scaler before fine-tuning

Token IDs, mapping, and scaler remain fixed. Each row must encode with no unsupported chemical tokens, fit the checkpoint length limit, round trip through BPE, and have 17 finite labels yielding finite z-scores with the model's scaler. Rejections are logged without truncation, vocabulary expansion, or zero-filled labels. Max_Absolute_ZScore indicates label scale, not guaranteed prediction quality.


In [ ]:
ready, compatibility_rejections = data_helpers.check_compatibility(labeled, predictor)
compatibility_rejections.to_csv(OUTPUT_DIR / "compatibility_rejections.csv", index=False)
print("Compatible with model assets:", len(ready), "; rejected:", len(compatibility_rejections))
assert not ready.empty, "No rows passed model-bundle compatibility; inspect rejection reasons"
display(ready[["NEW_Isomeric_SMILES", "Encoded_Length", "Max_Absolute_ZScore"]].head())


---

## 6 Split by parent group and export

Stereoisomers of one parent cannot cross Train/Val. At least two independent Parent_Group values are required. This example uses parent groups and the configured validation fraction. Export original-scale labels; z-scores are compatibility diagnostics, and later fine-tuning must retain the model's scaler.


In [ ]:
final_data = data_helpers.split_by_parent(ready, val_fraction=VAL_FRACTION, seed=EMBEDDING_SEED)
final_data.to_csv(OUTPUT_DIR / "finetuning_dataset.csv", index=False)
print(final_data["Split_Set"].value_counts())
print("Output:", OUTPUT_DIR / "finetuning_dataset.csv")
print("Preprocessing bundle:", predictor.bundle_fingerprint)
metadata.update({"expanded_rows": len(expanded), "optimized_rows": len(optimized),
                 "labeled_rows": len(labeled), "ready_rows": len(final_data), "val_fraction": VAL_FRACTION,
                 "split_counts": final_data["Split_Set"].value_counts().to_dict()})
(OUTPUT_DIR / "preparation_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


## Fine-Tuning with Prepared Data

Set DATASET in the [fine-tuning example](colab_finetuning.ipynb) to the exported finetuning_dataset.csv and select the same model bundle. Retain the model's tokenizer, mapping, and scaler. For already-labeled data, confirm definitions and units, then use step 5 for compatibility checks.

### References

- Stereoisomer enumeration](https://www.rdkit.org/docs/source/rdkit.Chem.EnumerateStereoisomers.html)
- MMFF and convergence status](https://www.rdkit.org/docs/source/rdkit.Chem.rdForceFieldHelpers.html)
- [FreeSASA calcCoord](https://freesasa.github.io/python/functions.html)
